## 03 — Pitch Distribution EDA

Visual exploration of how pitch selection varies across game contexts using June 2024 Statcast data enriched with 2023 pitcher/batter arsenal stats.

**Sections:**
1. Target Distribution — overall pitch type frequency and pitcher handedness split
2. Count-Situation Analysis — how balls and strikes shift pitch selection
3. Handedness Matchup — pitch mix across pitcher × batter handedness combinations
4. Game Context Features — pitch number, score differential, inning (`n_thruorder_pitcher` exploratory only)
5. Prior-Year Arsenal Usage — how well `pitch_usage_*` columns predict actual June 2024 usage

See **02_data_quality** for the column-level audit. See **04_feature_importance_ml** for feature selection experiments.

In [ ]:
%matplotlib inline

from pathlib import Path
import sys

import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import numpy as np
import pandas as pd
import seaborn as sns
import pybaseball
from pybaseball import cache, statcast

sys.path.append(str(Path("..").resolve()))
from utils.features.enrichment import enrich_statcast
from utils.features.feature_names import (
    BATTER_OUTCOME_COLUMNS,
    BATTER_USAGE_COLUMNS,
    CANDIDATE_GAME_STATE_COLUMNS,
    EDA_COLUMNS,
    LABEL_COLUMN,
    PITCHER_OUTCOME_COLUMNS,
    PITCHER_USAGE_COLUMNS,
    PITCH_TYPES,
)

DATA_DIR = Path("..") / "data" / "cache"
DATA_DIR.mkdir(parents=True, exist_ok=True)
cache.config.cache_directory = str(DATA_DIR.resolve())
cache.config.save()
cache.enable()

sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["figure.dpi"] = 120

print("pybaseball :", pybaseball.__version__)
print("pandas     :", pd.__version__)
print("cache dir  :", cache.config.cache_directory)

In [ ]:
SEASON = 2024
PRIOR  = SEASON - 1

df = statcast("2024-06-01", "2024-06-30")
print(f"Raw shape: {df.shape}")

df_enr = enrich_statcast(df, PRIOR)
print(f"Enriched shape: {df_enr.shape}")
if "pitch_usage_FF" in df_enr.columns:
    print(f"Pitcher usage coverage (pitch_usage_FF non-null): {df_enr['pitch_usage_FF'].notna().mean():.1%}")

In [ ]:
available    = [c for c in EDA_COLUMNS if c in df_enr.columns]
missing_cols = [c for c in EDA_COLUMNS if c not in df_enr.columns]

model_df = df_enr[available + [LABEL_COLUMN]].copy()
model_df = model_df[model_df[LABEL_COLUMN].isin(PITCH_TYPES)].reset_index(drop=True)

print(f"model_df shape       : {model_df.shape}")
print(f"EDA_COLUMNS total    : {len(EDA_COLUMNS)}")
print(f"Found in enriched frame: {len(available)}")
if missing_cols:
    preview = missing_cols[:5]
    ellipsis = "..." if len(missing_cols) > 5 else ""
    print(f"Missing ({len(missing_cols)})           : {preview}{ellipsis}")
print(f"\nPitch type distribution:\n{model_df[LABEL_COLUMN].value_counts()}")